# CatBoost

In [1]:
import os
os.chdir("E:\Data Science\ML\Project")

In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_auc_score)

## load the dataset

In [3]:
df = pd.read_csv("data\ChurnGuard_processed.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9259 entries, 0 to 9258
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                9259 non-null   int64  
 1   gender             9259 non-null   object 
 2   tenure_months      9259 non-null   int64  
 3   contract_type      9259 non-null   object 
 4   monthly_charges    9259 non-null   float64
 5   total_charges      9259 non-null   float64
 6   internet_service   9259 non-null   object 
 7   payment_method     9259 non-null   object 
 8   support_calls      9259 non-null   int64  
 9   late_payments      9259 non-null   int64  
 10  online_security    9259 non-null   object 
 11  tech_support       9259 non-null   object 
 12  streaming_service  9259 non-null   object 
 13  senior_citizen     9259 non-null   int64  
 14  family_members     9259 non-null   int64  
 15  churn              9259 non-null   int64  
dtypes: float64(2), int64(7),

In [4]:
X = df.drop(columns="churn")
y = df.churn

## Data Preprocessing & CatBoost Pipeline

In [5]:
num_var = X.select_dtypes(exclude="object").columns
cat_var = X.select_dtypes(include="object").columns

In [6]:
preprocessing = ColumnTransformer([("num", "passthrough", num_var), ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_var)])
model = Pipeline([("preprocessing", preprocessing), ("classifier", CatBoostClassifier(loss_function="Logloss", eval_metric="AUC", random_seed=42, verbose=0, thread_count=-1))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.27, random_state=57)

## Define GridSearch parameters

In [7]:
param_grid = {
    "classifier__depth": [4, 6, 8],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__iterations": [100, 200],
    "classifier__l2_leaf_reg": [1, 3, 5]
}

## GridSearchCV

In [8]:
grid_cb = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=1, return_train_score=True)
grid_cb.fit(X_train, y_train)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


,estimator,Pipeline(step... verbose=0))])
,param_grid,"{'classifier__depth': [4, 6, ...], 'classifier__iterations': [100, 200], 'classifier__l2_leaf_reg': [1, 3, ...], 'classifier__learning_rate': [0.03, 0.05, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


## Best Parameters

In [9]:
print("Best Parameters:")
print(grid_cb.best_params_)

print("\nBest CV Accuracy:")
print(grid_cb.best_score_)

Best Parameters:
{'classifier__depth': 4, 'classifier__iterations': 100, 'classifier__l2_leaf_reg': 3, 'classifier__learning_rate': 0.05}

Best CV Accuracy:
0.7780736600983712


## Model Performance Evaluation

In [10]:
best_cb = grid_cb.best_estimator_

In [11]:
y_pred = best_cb.predict(X_test)
y_pred_prob = best_cb.predict_proba(X_test)[:, 1]

In [12]:
confusion_matrix(y_test, y_pred)

array([[1889,   21],
       [ 546,   44]])

In [13]:
accuracy_score(y_test, y_pred)

0.7732

In [14]:
roc_auc_score(y_test, y_pred_prob)

0.7094436063537137

In [15]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.99      0.87      1910
           1       0.68      0.07      0.13       590

    accuracy                           0.77      2500
   macro avg       0.73      0.53      0.50      2500
weighted avg       0.75      0.77      0.70      2500

